# 01 · Stage 1 — DR-supervised training of the CFP, UWF and descriptor encoders

This is the notebook that produced the Stage-1 encoder checkpoint (`stage6_full_effnet_b0_best.pt`) used by notebook 02. Its code and saved outputs are unchanged from the original run (`P2_model_v5(Final).ipynb`); only these markdown cells and the environment cell below were added.

**What it does.** Two ImageNet-initialised EfficientNet-B0 encoders (CFP, UWF) with 512-d projections and a descriptor encoder (30 → 256 → 512) are trained jointly on the referable-DR label with a temporary softmax gate and classifier, for 5 epochs, and the epoch with the best validation F1 is saved.

**Split used here.** Samples with a CFP or a UWF image (16,451 of 17,224) are split 70/15/15, stratified, seed 42: train 11,515, validation 2,468, test 2,468. Notebook 02 draws its own split over all 17,224 samples, so the two partitions differ: 1,740 of the 2,584 test samples of notebook 02 are in the training partition of this notebook (see notebook 03).

**Run history.** Cells 3–5 trained the model on a T4 GPU. The evaluation in cell 6 was interrupted by a Colab runtime reset before it printed results; cells 7–11 therefore reload the checkpoint and repeat the evaluation without test-time augmentation. Their result (`stage6_resumed_final_metrics.csv`) is the Stage-1 baseline reported in the article.

**Inputs:** `DatasetRT/manifest.csv` (notebook 00), `DatasetRT/metadata.csv` (from `data/`), MMRDR images.  
**Outputs (`stage1_outputs/`):** `stage6_full_effnet_b0_best.pt`, `stage6_full_effnet_b0_training_history.csv`, `stage6_full_dataset_split_summary.csv`, `stage6_resumed_final_metrics.csv`, `stage6_resumed_threshold_search.csv`.

In [ ]:
# Added for reproducibility (not part of the original run): records the computing environment.
import platform, sys, os
import torch, sklearn, numpy as np, pandas as pd
try:
    import timm; timm_version = timm.__version__
except ImportError:
    timm_version = 'not installed yet (installed in a later cell)'
print('Python:', sys.version.split()[0], '| OS:', platform.platform(), '| CPU cores:', os.cpu_count())
try:
    import psutil; print('RAM (GB):', round(psutil.virtual_memory().total / 1e9, 1))
except ImportError:
    pass
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
print('torch', torch.__version__, '| timm', timm_version, '| scikit-learn', sklearn.__version__, '| numpy', np.__version__, '| pandas', pd.__version__)

## Mount Google Drive and check the inputs

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
from pathlib import Path
import torch

PROJECT_ROOT = Path("/content/drive/MyDrive/MMRDR_Project")
DATA_ROOT = PROJECT_ROOT / "DatasetRT"
OUTPUT_DIR = PROJECT_ROOT / "stage1_outputs"

print("DATA_ROOT exists:", DATA_ROOT.exists())
print("manifest exists:", (DATA_ROOT / "manifest.csv").exists())
print("metadata exists:", (DATA_ROOT / "metadata.csv").exists())
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)

DATA_ROOT exists: True
manifest exists: True
metadata exists: True
CUDA: True
GPU: Tesla T4


## Configuration and Stage-1 split
`BATCH_SIZE = 8`, `EPOCHS = 5`, `LR = 1e-4`, backbone `tf_efficientnet_b0`, seed 42. Only samples with a CFP or a UWF image are kept; OCT-only samples are not used in this stage.

In [ ]:
# ============================================================
# STAGE 6: Full-Dataset EfficientNet-B0 Gated Fusion
# Cell 1: Setup + Data Split
# ============================================================

!pip -q install timm

from pathlib import Path
import os, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import timm
import matplotlib.pyplot as plt
import seaborn as sns

from PIL import Image, ImageOps, ImageEnhance
from tqdm.auto import tqdm
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, brier_score_loss,
    confusion_matrix, classification_report, roc_curve,
    precision_recall_curve
)

PROJECT_ROOT = Path("/content/drive/MyDrive/MMRDR_Project")
DATA_ROOT = PROJECT_ROOT / "DatasetRT"
OUTPUT_DIR = PROJECT_ROOT / "stage1_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42
IMAGE_SIZE = 224
BATCH_SIZE = 8
EPOCHS = 5
LR = 1e-4
BACKBONE = "tf_efficientnet_b0"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print("Device:", DEVICE)
print("DATA_ROOT exists:", DATA_ROOT.exists())

manifest = pd.read_csv(DATA_ROOT / "manifest.csv")
metadata = pd.read_csv(DATA_ROOT / "metadata.csv")

df = manifest.merge(metadata, on=["sample_id", "patient_id"], how="left")
df = df.dropna(subset=["dr_grade"]).copy()
df["dr_grade"] = df["dr_grade"].astype(int)
df["referable_dr"] = (df["dr_grade"] >= 2).astype(int)

# Keep samples with at least one retinal image among final modalities
df = df[df["fundus_path"].notna() | df["uwf_path"].notna()].copy()

print("\nFull usable dataset:", df.shape)
print(df["referable_dr"].value_counts().sort_index())

train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    random_state=SEED,
    stratify=df["referable_dr"],
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_df["referable_dr"],
)

print("\nTrain:", train_df.shape)
print(train_df["referable_dr"].value_counts().sort_index())

print("\nVal:", val_df.shape)
print(val_df["referable_dr"].value_counts().sort_index())

print("\nTest:", test_df.shape)
print(test_df["referable_dr"].value_counts().sort_index())

split_summary = pd.DataFrame([
    {"split": "train", "n": len(train_df), "non_referable": int((train_df["referable_dr"] == 0).sum()), "referable": int((train_df["referable_dr"] == 1).sum())},
    {"split": "val", "n": len(val_df), "non_referable": int((val_df["referable_dr"] == 0).sum()), "referable": int((val_df["referable_dr"] == 1).sum())},
    {"split": "test", "n": len(test_df), "non_referable": int((test_df["referable_dr"] == 0).sum()), "referable": int((test_df["referable_dr"] == 1).sum())},
])

split_summary.to_csv(OUTPUT_DIR / "stage6_full_dataset_split_summary.csv", index=False)
display(split_summary)

Device: cuda
DATA_ROOT exists: True

Full usable dataset: (16451, 32)
referable_dr
0    10650
1     5801
Name: count, dtype: int64

Train: (11515, 32)
referable_dr
0    7455
1    4060
Name: count, dtype: int64

Val: (2468, 32)
referable_dr
0    1597
1     871
Name: count, dtype: int64

Test: (2468, 32)
referable_dr
0    1598
1     870
Name: count, dtype: int64


,split,n,non_referable,referable
0,train,11515,7455,4060
1,val,2468,1597,871
2,test,2468,1598,870


## Descriptor preprocessing and data loaders
Numeric descriptors: median imputation + z-score; categorical: most-frequent imputation + one-hot (fitted on the training split; 30 features). Images: RGB, 224 × 224, ImageNet normalisation; training augmentation: horizontal flip (p 0.5), brightness and contrast ±10 % (p 0.35 each). A weighted random sampler balances the classes.

In [ ]:
# ============================================================
# Cell 2: Metadata preprocessing + DataLoaders
# ============================================================

numeric_cols = [
    "age", "diabetes_duration", "hba1c", "systolic_bp", "diastolic_bp",
    "bmi", "visual_acuity", "intraocular_pressure", "metadata_is_synthetic"
]

categorical_cols = [
    "gender", "ethnicity", "diabetes_type", "hypertension", "hyperlipidemia",
    "kidney_disease", "smoking_status", "alcohol_status"
]

numeric_cols = [c for c in numeric_cols if c in train_df.columns]
categorical_cols = [c for c in categorical_cols if c in train_df.columns]

try:
    onehot = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    onehot = OneHotEncoder(handle_unknown="ignore", sparse=False)

metadata_preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), numeric_cols),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", onehot),
        ]), categorical_cols),
    ],
)

train_meta = metadata_preprocessor.fit_transform(train_df).astype(np.float32)
val_meta = metadata_preprocessor.transform(val_df).astype(np.float32)
test_meta = metadata_preprocessor.transform(test_df).astype(np.float32)

metadata_dim = train_meta.shape[1]
print("metadata_dim:", metadata_dim)

def load_image(path, image_size=224, augment=False):
    if pd.isna(path) or not Path(str(path)).exists():
        return torch.zeros(3, image_size, image_size)

    img = Image.open(path).convert("RGB")
    img = img.resize((image_size, image_size))

    if augment:
        if random.random() < 0.5:
            img = ImageOps.mirror(img)
        if random.random() < 0.35:
            img = ImageEnhance.Brightness(img).enhance(random.uniform(0.90, 1.10))
        if random.random() < 0.35:
            img = ImageEnhance.Contrast(img).enhance(random.uniform(0.90, 1.10))

    arr = np.asarray(img).astype(np.float32) / 255.0
    arr = (arr - np.array([0.485, 0.456, 0.406])) / np.array([0.229, 0.224, 0.225])

    return torch.from_numpy(arr).permute(2, 0, 1).float()

class FullBinaryDataset(Dataset):
    def __init__(self, dataframe, metadata_array, augment=False):
        self.df = dataframe.reset_index(drop=True)
        self.meta = metadata_array
        self.augment = augment

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        return {
            "fundus": load_image(row.get("fundus_path", np.nan), IMAGE_SIZE, self.augment),
            "uwf": load_image(row.get("uwf_path", np.nan), IMAGE_SIZE, self.augment),
            "metadata": torch.tensor(self.meta[idx], dtype=torch.float32),
            "label": torch.tensor(int(row["referable_dr"]), dtype=torch.long),
        }

train_dataset = FullBinaryDataset(train_df, train_meta, augment=True)
val_dataset = FullBinaryDataset(val_df, val_meta, augment=False)
test_dataset = FullBinaryDataset(test_df, test_meta, augment=False)

train_labels = train_df["referable_dr"].values
class_counts = np.bincount(train_labels, minlength=2)
class_weights = len(train_labels) / (2.0 * np.maximum(class_counts, 1))
sample_weights = class_weights[train_labels]

sampler = WeightedRandomSampler(
    weights=torch.DoubleTensor(sample_weights),
    num_samples=len(sample_weights),
    replacement=True,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=sampler,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
)

print("class_counts:", class_counts)
print("class_weights:", class_weights)
print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))
print("Test batches:", len(test_loader))

batch = next(iter(train_loader))
print("fundus:", batch["fundus"].shape)
print("uwf:", batch["uwf"].shape)
print("metadata:", batch["metadata"].shape)
print("label:", batch["label"].shape)

metadata_dim: 30
class_counts: [7455 4060]
class_weights: [0.77230047 1.41810345]
Train batches: 1440
Val batches: 309
Test batches: 309
fundus: torch.Size([8, 3, 224, 224])
uwf: torch.Size([8, 3, 224, 224])
metadata: torch.Size([8, 30])
label: torch.Size([8])


## Model and training
Gated fusion of the CFP, UWF and descriptor embeddings; AdamW (weight decay 1e-4), cosine schedule, mixed precision, gradient clipping 1.0, class-weighted cross-entropy with label smoothing 0.03. The checkpoint with the best validation F1 (argmax) is saved (epoch 4, F1 0.8718).

In [ ]:
# ============================================================
# Cell 3: Model + Training
# ============================================================

class Stage6EfficientNetFusion(nn.Module):
    def __init__(self, metadata_dim, backbone="tf_efficientnet_b0", embedding_dim=512):
        super().__init__()

        self.fundus_encoder = timm.create_model(
            backbone,
            pretrained=True,
            num_classes=0,
            global_pool="avg",
        )

        self.uwf_encoder = timm.create_model(
            backbone,
            pretrained=True,
            num_classes=0,
            global_pool="avg",
        )

        image_dim = self.fundus_encoder.num_features

        self.fundus_proj = nn.Sequential(
            nn.Linear(image_dim, embedding_dim),
            nn.LayerNorm(embedding_dim),
            nn.GELU(),
            nn.Dropout(0.30),
        )

        self.uwf_proj = nn.Sequential(
            nn.Linear(image_dim, embedding_dim),
            nn.LayerNorm(embedding_dim),
            nn.GELU(),
            nn.Dropout(0.30),
        )

        self.metadata_encoder = nn.Sequential(
            nn.Linear(metadata_dim, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(0.25),
            nn.Linear(256, embedding_dim),
            nn.LayerNorm(embedding_dim),
            nn.GELU(),
        )

        self.gate = nn.Sequential(
            nn.Linear(embedding_dim * 3, 256),
            nn.GELU(),
            nn.Dropout(0.25),
            nn.Linear(256, 3),
        )

        self.classifier = nn.Sequential(
            nn.Linear(embedding_dim, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(0.30),
            nn.Linear(256, 2),
        )

    def forward(self, batch):
        f = self.fundus_proj(self.fundus_encoder(batch["fundus"]))
        u = self.uwf_proj(self.uwf_encoder(batch["uwf"]))
        m = self.metadata_encoder(batch["metadata"])

        gates = torch.softmax(self.gate(torch.cat([f, u, m], dim=1)), dim=1)
        fused = gates[:, 0:1] * f + gates[:, 1:2] * u + gates[:, 2:3] * m

        return {
            "logits": self.classifier(fused),
            "embedding": fused,
            "gates": gates,
        }

def move_to_device(batch):
    return {k: v.to(DEVICE) if torch.is_tensor(v) else v for k, v in batch.items()}

model = Stage6EfficientNetFusion(
    metadata_dim=metadata_dim,
    backbone=BACKBONE,
).to(DEVICE)

criterion = nn.CrossEntropyLoss(
    weight=torch.tensor(class_weights, dtype=torch.float32).to(DEVICE),
    label_smoothing=0.03,
)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE == "cuda"))

def evaluate_argmax(model, loader):
    model.eval()

    all_probs = []
    all_y = []

    with torch.no_grad():
        for batch in tqdm(loader, desc="Evaluating", leave=False):
            batch = move_to_device(batch)
            out = model(batch)
            probs = torch.softmax(out["logits"], dim=1)

            all_probs.append(probs.cpu().numpy())
            all_y.append(batch["label"].cpu().numpy())

    probs = np.concatenate(all_probs)
    y = np.concatenate(all_y)
    preds = probs.argmax(axis=1)

    return {
        "accuracy": accuracy_score(y, preds),
        "precision": precision_score(y, preds, zero_division=0),
        "recall": recall_score(y, preds, zero_division=0),
        "f1": f1_score(y, preds, zero_division=0),
        "roc_auc": roc_auc_score(y, probs[:, 1]),
        "pr_auc": average_precision_score(y, probs[:, 1]),
    }

best_f1 = -1
best_path = OUTPUT_DIR / "stage6_full_effnet_b0_best.pt"
history = []

print("Training Stage 6 on:", DEVICE)

for epoch in range(1, EPOCHS + 1):
    model.train()
    losses = []

    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}")

    for batch in pbar:
        batch = move_to_device(batch)
        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=(DEVICE == "cuda")):
            out = model(batch)
            loss = criterion(out["logits"], batch["label"])

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()

        losses.append(loss.item())
        pbar.set_postfix(loss=float(np.mean(losses)))

    scheduler.step()

    val_metrics = evaluate_argmax(model, val_loader)
    train_loss = float(np.mean(losses))

    row = {"epoch": epoch, "train_loss": train_loss, **val_metrics}
    history.append(row)

    print(
        f"Epoch {epoch}/{EPOCHS} | "
        f"loss={train_loss:.4f} | "
        f"acc={val_metrics['accuracy']:.4f} | "
        f"f1={val_metrics['f1']:.4f} | "
        f"auc={val_metrics['roc_auc']:.4f}"
    )

    if val_metrics["f1"] > best_f1:
        best_f1 = val_metrics["f1"]

        torch.save(
            {
                "model_state": model.state_dict(),
                "metadata_dim": metadata_dim,
                "backbone": BACKBONE,
                "image_size": IMAGE_SIZE,
                "class_weights": class_weights,
                "seed": SEED,
            },
            best_path,
        )

        print("Saved best:", best_path)

pd.DataFrame(history).to_csv(
    OUTPUT_DIR / "stage6_full_effnet_b0_training_history.csv",
    index=False,
)

print("Training complete")
print("Best validation F1:", best_f1)
print("Best checkpoint:", best_path)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

Training Stage 6 on: cuda


/tmp/ipykernel_466/3545652543.py:93: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE == "cuda"))


Epoch 1/5:   0%|          | 0/1440 [00:00<?, ?it/s]

...

Epoch 4/5 | loss=0.2237 | acc=0.9088 | f1=0.8718 | auc=0.9664
Saved best: /content/drive/MyDrive/MMRDR_Project/stage1_outputs/stage6_full_effnet_b0_best.pt


Epoch 5/5:   0%|          | 0/1440 [00:00<?, ?it/s]

/tmp/ipykernel_466/3545652543.py:139: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(DEVICE == "cuda")):


Evaluating:   0%|          | 0/309 [00:00<?, ?it/s]

Epoch 5/5 | loss=0.1992 | acc=0.9064 | f1=0.8710 | auc=0.9663
Training complete
Best validation F1: 0.8717948717948718
Best checkpoint: /content/drive/MyDrive/MMRDR_Project/stage1_outputs/stage6_full_effnet_b0_best.pt


## Evaluation with test-time augmentation (interrupted)
This cell was interrupted by a Colab runtime reset and printed no results. The evaluation is repeated below.

In [ ]:
# ============================================================
# Cell 4: Final full test with TTA + validation threshold
# PyTorch 2.6+ checkpoint loading fixed
# ============================================================

# Load checkpoint safely from your own trusted training run
try:
    checkpoint = torch.load(best_path, map_location=DEVICE, weights_only=False)
except TypeError:
    checkpoint = torch.load(best_path, map_location=DEVICE)

print("Checkpoint loaded successfully")
print("Checkpoint keys:", checkpoint.keys())

model = Stage6EfficientNetFusion(
    metadata_dim=checkpoint["metadata_dim"],
    backbone=checkpoint["backbone"],
).to(DEVICE)

model.load_state_dict(checkpoint["model_state"])
model.eval()

print("Best Stage 6 model loaded")

def predict_tta(model, loader):
    model.eval()

    probs_all = []
    y_all = []
    gates_all = []

    with torch.no_grad():
        for batch in tqdm(loader, desc="TTA prediction", leave=False):
            batch = move_to_device(batch)

            out1 = model(batch)
            p1 = torch.softmax(out1["logits"], dim=1)

            batch_flip = {
                k: v.clone() if torch.is_tensor(v) else v
                for k, v in batch.items()
            }

            batch_flip["fundus"] = torch.flip(batch_flip["fundus"], dims=[3])
            batch_flip["uwf"] = torch.flip(batch_flip["uwf"], dims=[3])

            out2 = model(batch_flip)
            p2 = torch.softmax(out2["logits"], dim=1)

            probs = (p1 + p2) / 2.0

            probs_all.append(probs.cpu().numpy())
            y_all.append(batch["label"].cpu().numpy())
            gates_all.append(out1["gates"].cpu().numpy())

    return (
        np.concatenate(y_all),
        np.concatenate(probs_all),
        np.concatenate(gates_all),
    )

# Predict validation and test sets with TTA
y_val, p_val, _ = predict_tta(model, val_loader)
y_test, p_test, gate_weights = predict_tta(model, test_loader)

print("Validation predictions:", p_val.shape)
print("Test predictions:", p_test.shape)

# Tune threshold on validation set only
rows = []

for threshold in np.linspace(0.05, 0.95, 181):
    preds = (p_val[:, 1] >= threshold).astype(int)

    rows.append({
        "threshold": threshold,
        "accuracy": accuracy_score(y_val, preds),
        "precision": precision_score(y_val, preds, zero_division=0),
        "recall": recall_score(y_val, preds, zero_division=0),
        "f1": f1_score(y_val, preds, zero_division=0),
    })

threshold_df = pd.DataFrame(rows)

best_row = threshold_df.loc[threshold_df["f1"].idxmax()]
best_threshold = float(best_row["threshold"])

print("Best validation threshold:")
display(best_row)

# Final test metrics using validation-selected threshold
test_probs = p_test[:, 1]
test_preds = (test_probs >= best_threshold).astype(int)

stage6_metrics = {
    "model": "Stage 6: Full-data EfficientNet-B0 gated fusion + TTA + threshold",
    "threshold": best_threshold,
    "test_n": len(y_test),
    "accuracy": accuracy_score(y_test, test_preds),
    "precision": precision_score(y_test, test_preds, zero_division=0),
    "recall": recall_score(y_test, test_preds, zero_division=0),
    "f1": f1_score(y_test, test_preds, zero_division=0),
    "roc_auc": roc_auc_score(y_test, test_probs),
    "pr_auc": average_precision_score(y_test, test_probs),
    "brier_score": brier_score_loss(y_test, test_probs),
}

print("Stage 6 final full-test metrics:")
print(stage6_metrics)

pd.DataFrame([stage6_metrics]).to_csv(
    OUTPUT_DIR / "stage6_full_effnet_b0_test_metrics.csv",
    index=False,
)

threshold_df.to_csv(
    OUTPUT_DIR / "stage6_validation_threshold_search.csv",
    index=False,
)

target_names = ["Non-referable DR", "Referable DR"]

report = classification_report(
    y_test,
    test_preds,
    target_names=target_names,
    digits=4,
)

print("\nClassification report:")
print(report)

with open(OUTPUT_DIR / "stage6_full_effnet_b0_classification_report.txt", "w") as f:
    f.write(report)

# Confusion matrix
cm = confusion_matrix(y_test, test_preds)

plt.figure(figsize=(6, 5), dpi=160)
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=target_names,
    yticklabels=target_names,
)
plt.title("Stage 6 Full-Test Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "stage6_full_effnet_b0_confusion_matrix.png",
    bbox_inches="tight",
)
plt.show()

# ROC curve
fpr, tpr, _ = roc_curve(y_test, test_probs)
roc_auc = roc_auc_score(y_test, test_probs)

plt.figure(figsize=(6, 5), dpi=160)
plt.plot(fpr, tpr, linewidth=2, label=f"ROC-AUC = {roc_auc:.3f}")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
plt.title("Stage 6 Full-Test ROC Curve")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "stage6_full_effnet_b0_roc_curve.png",
    bbox_inches="tight",
)
plt.show()

# Precision-recall curve
precision_curve, recall_curve, _ = precision_recall_curve(y_test, test_probs)
pr_auc = average_precision_score(y_test, test_probs)

plt.figure(figsize=(6, 5), dpi=160)
plt.plot(
    recall_curve,
    precision_curve,
    linewidth=2,
    label=f"PR-AUC = {pr_auc:.3f}",
)
plt.title("Stage 6 Full-Test Precision-Recall Curve")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.legend(loc="lower left")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "stage6_full_effnet_b0_pr_curve.png",
    bbox_inches="tight",
)
plt.show()

# Gate weights
gate_df = pd.DataFrame(
    gate_weights,
    columns=["fundus_weight", "uwf_weight", "metadata_weight"],
)

gate_df.to_csv(
    OUTPUT_DIR / "stage6_full_effnet_b0_gate_weights.csv",
    index=False,
)

gate_df.describe().to_csv(
    OUTPUT_DIR / "stage6_full_effnet_b0_gate_weight_summary.csv",
)

print("Saved Stage 6 full-dataset outputs in:")
print(OUTPUT_DIR)

Checkpoint loaded successfully
Checkpoint keys: dict_keys(['model_state', 'metadata_dim', 'backbone', 'image_size', 'class_weights', 'seed'])
Best Stage 6 model loaded


TTA prediction:   0%|          | 0/309 [00:00<?, ?it/s]

TTA prediction:   0%|          | 0/309 [00:00<?, ?it/s]

## Re-evaluation of the saved Stage-1 checkpoint (after the runtime reset)
The split is re-drawn with the same code and seed (identical partition), the checkpoint is reloaded, the threshold is chosen on the validation split (181 values in [0.05, 0.95], maximising F1) and the test split is scored once, without test-time augmentation. This run used the CPU runtime.

In [1]:
from google.colab import drive
from pathlib import Path
import torch

drive.mount("/content/drive")

PROJECT_ROOT = Path("/content/drive/MyDrive/MMRDR_Project")
DATA_ROOT = PROJECT_ROOT / "DatasetRT"
OUTPUT_DIR = PROJECT_ROOT / "stage1_outputs"
BEST_PATH = OUTPUT_DIR / "stage6_full_effnet_b0_best.pt"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42
IMAGE_SIZE = 224
EVAL_BATCH_SIZE = 16 if DEVICE == "cuda" else 4

print("Device:", DEVICE)
print("DATA_ROOT:", DATA_ROOT.exists())
print("manifest:", (DATA_ROOT / "manifest.csv").exists())
print("metadata:", (DATA_ROOT / "metadata.csv").exists())
print("checkpoint:", BEST_PATH.exists())

Mounted at /content/drive
Device: cpu
DATA_ROOT: True
manifest: True
metadata: True
checkpoint: True


In [2]:
!pip -q install timm

import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import timm
from PIL import Image
from tqdm.auto import tqdm

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score, brier_score_loss, classification_report

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

manifest = pd.read_csv(DATA_ROOT / "manifest.csv")
metadata = pd.read_csv(DATA_ROOT / "metadata.csv")

df = manifest.merge(metadata, on=["sample_id", "patient_id"], how="left")
df = df.dropna(subset=["dr_grade"]).copy()
df["dr_grade"] = df["dr_grade"].astype(int)
df["referable_dr"] = (df["dr_grade"] >= 2).astype(int)
df = df[df["fundus_path"].notna() | df["uwf_path"].notna()].copy()

train_df, temp_df = train_test_split(
    df, test_size=0.30, random_state=SEED, stratify=df["referable_dr"]
)

val_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=SEED, stratify=temp_df["referable_dr"]
)

print("Train:", train_df.shape, train_df["referable_dr"].value_counts().to_dict())
print("Val:", val_df.shape, val_df["referable_dr"].value_counts().to_dict())
print("Test:", test_df.shape, test_df["referable_dr"].value_counts().to_dict())

Train: (11515, 32) {0: 7455, 1: 4060}
Val: (2468, 32) {0: 1597, 1: 871}
Test: (2468, 32) {0: 1598, 1: 870}


In [3]:
numeric_cols = [
    "age", "diabetes_duration", "hba1c", "systolic_bp", "diastolic_bp",
    "bmi", "visual_acuity", "intraocular_pressure", "metadata_is_synthetic"
]

categorical_cols = [
    "gender", "ethnicity", "diabetes_type", "hypertension", "hyperlipidemia",
    "kidney_disease", "smoking_status", "alcohol_status"
]

numeric_cols = [c for c in numeric_cols if c in train_df.columns]
categorical_cols = [c for c in categorical_cols if c in train_df.columns]

try:
    onehot = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    onehot = OneHotEncoder(handle_unknown="ignore", sparse=False)

metadata_preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]), numeric_cols),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", onehot),
    ]), categorical_cols),
])

train_meta = metadata_preprocessor.fit_transform(train_df).astype(np.float32)
val_meta = metadata_preprocessor.transform(val_df).astype(np.float32)
test_meta = metadata_preprocessor.transform(test_df).astype(np.float32)

print("metadata_dim:", train_meta.shape[1])

metadata_dim: 30


In [4]:
class ResumeStage6Net(nn.Module):
    def __init__(self, metadata_dim, backbone):
        super().__init__()

        self.fundus_encoder = timm.create_model(backbone, pretrained=False, num_classes=0, global_pool="avg")
        self.uwf_encoder = timm.create_model(backbone, pretrained=False, num_classes=0, global_pool="avg")

        image_dim = self.fundus_encoder.num_features
        emb = 512

        self.fundus_proj = nn.Sequential(nn.Linear(image_dim, emb), nn.LayerNorm(emb), nn.GELU(), nn.Dropout(0.30))
        self.uwf_proj = nn.Sequential(nn.Linear(image_dim, emb), nn.LayerNorm(emb), nn.GELU(), nn.Dropout(0.30))

        self.metadata_encoder = nn.Sequential(
            nn.Linear(metadata_dim, 256), nn.LayerNorm(256), nn.GELU(), nn.Dropout(0.25),
            nn.Linear(256, emb), nn.LayerNorm(emb), nn.GELU()
        )

        self.gate = nn.Sequential(nn.Linear(emb * 3, 256), nn.GELU(), nn.Dropout(0.25), nn.Linear(256, 3))

        self.classifier = nn.Sequential(
            nn.Linear(emb, 256), nn.LayerNorm(256), nn.GELU(), nn.Dropout(0.30), nn.Linear(256, 2)
        )

    def forward(self, batch):
        f = self.fundus_proj(self.fundus_encoder(batch["fundus"]))
        u = self.uwf_proj(self.uwf_encoder(batch["uwf"]))
        m = self.metadata_encoder(batch["metadata"])

        gates = torch.softmax(self.gate(torch.cat([f, u, m], dim=1)), dim=1)
        fused = gates[:, 0:1] * f + gates[:, 1:2] * u + gates[:, 2:3] * m

        return {"logits": self.classifier(fused), "gates": gates}


checkpoint = torch.load(BEST_PATH, map_location=DEVICE, weights_only=False)

print("Checkpoint metadata_dim:", checkpoint["metadata_dim"])
print("Checkpoint backbone:", checkpoint["backbone"])

model = ResumeStage6Net(
    checkpoint["metadata_dim"],
    checkpoint["backbone"]
).to(DEVICE)

model.load_state_dict(checkpoint["model_state"])
model.eval()

print("Stage 6 checkpoint loaded")

Checkpoint metadata_dim: 30
Checkpoint backbone: tf_efficientnet_b0
Stage 6 checkpoint loaded


In [5]:
def load_img(path):
    if pd.isna(path) or not Path(str(path)).exists():
        return torch.zeros(3, IMAGE_SIZE, IMAGE_SIZE)

    img = Image.open(path).convert("RGB").resize((IMAGE_SIZE, IMAGE_SIZE))
    arr = np.asarray(img).astype(np.float32) / 255.0
    arr = (arr - np.array([0.485, 0.456, 0.406])) / np.array([0.229, 0.224, 0.225])
    return torch.from_numpy(arr).permute(2, 0, 1).float()


def predict_manual(dataframe, meta_array):
    probs_all = []
    labels_all = []

    dataframe = dataframe.reset_index(drop=True)

    with torch.no_grad():
        for start in tqdm(range(0, len(dataframe), EVAL_BATCH_SIZE), desc="Predicting"):
            end = min(start + EVAL_BATCH_SIZE, len(dataframe))
            part = dataframe.iloc[start:end]

            fundus = torch.stack([load_img(r.get("fundus_path", np.nan)) for _, r in part.iterrows()]).to(DEVICE)
            uwf = torch.stack([load_img(r.get("uwf_path", np.nan)) for _, r in part.iterrows()]).to(DEVICE)
            meta = torch.tensor(meta_array[start:end], dtype=torch.float32).to(DEVICE)
            labels = part["referable_dr"].astype(int).values

            out = model({"fundus": fundus, "uwf": uwf, "metadata": meta})
            probs = torch.softmax(out["logits"], dim=1).cpu().numpy()

            probs_all.append(probs)
            labels_all.append(labels)

    return np.concatenate(labels_all), np.concatenate(probs_all)


y_val, p_val = predict_manual(val_df, val_meta)
y_test, p_test = predict_manual(test_df, test_meta)

rows = []
for threshold in np.linspace(0.05, 0.95, 181):
    pred = (p_val[:, 1] >= threshold).astype(int)
    rows.append({
        "threshold": threshold,
        "f1": f1_score(y_val, pred, zero_division=0),
        "accuracy": accuracy_score(y_val, pred),
        "precision": precision_score(y_val, pred, zero_division=0),
        "recall": recall_score(y_val, pred, zero_division=0),
    })

threshold_df = pd.DataFrame(rows)
best = threshold_df.loc[threshold_df["f1"].idxmax()]
best_threshold = float(best["threshold"])

print("Best validation threshold:")
display(best)

test_probs = p_test[:, 1]
test_pred = (test_probs >= best_threshold).astype(int)

metrics = {
    "threshold": best_threshold,
    "test_n": len(y_test),
    "accuracy": accuracy_score(y_test, test_pred),
    "precision": precision_score(y_test, test_pred, zero_division=0),
    "recall": recall_score(y_test, test_pred, zero_division=0),
    "f1": f1_score(y_test, test_pred, zero_division=0),
    "roc_auc": roc_auc_score(y_test, test_probs),
    "pr_auc": average_precision_score(y_test, test_probs),
    "brier_score": brier_score_loss(y_test, test_probs),
}

print("Stage 6 FINAL metrics:")
print(metrics)

print("\nClassification report:")
print(classification_report(y_test, test_pred, target_names=["Non-referable DR", "Referable DR"], digits=4))

pd.DataFrame([metrics]).to_csv(OUTPUT_DIR / "stage6_resumed_final_metrics.csv", index=False)
threshold_df.to_csv(OUTPUT_DIR / "stage6_resumed_threshold_search.csv", index=False)

print("Saved:", OUTPUT_DIR / "stage6_resumed_final_metrics.csv")

Predicting:   0%|          | 0/617 [00:00<?, ?it/s]

Predicting:   0%|          | 0/617 [00:00<?, ?it/s]

Best validation threshold:


,72
threshold,0.410000
f1,0.873940
accuracy,0.909643
precision,0.860802
recall,0.887486


Stage 6 FINAL metrics:
{'threshold': 0.4099999999999999, 'test_n': 2468, 'accuracy': 0.9246353322528363, 'precision': 0.8834080717488789, 'recall': 0.9057471264367816, 'f1': 0.8944381384790011, 'roc_auc': np.float64(0.9725123358220764), 'pr_auc': np.float64(0.9540961494302), 'brier_score': np.float64(0.06303333109938211)}

Classification report:
                  precision    recall  f1-score   support

Non-referable DR     0.9480    0.9349    0.9414      1598
    Referable DR     0.8834    0.9057    0.8944       870

        accuracy                         0.9246      2468
       macro avg     0.9157    0.9203    0.9179      2468
    weighted avg     0.9252    0.9246    0.9248      2468

Saved: /content/drive/MyDrive/MMRDR_Project/stage1_outputs/stage6_resumed_final_metrics.csv
